# CogMem Phase 2 — Q-STaR Training (Generator + Verifier DoRA)

Two-stage training pipeline for BigCodeBench:

1. **Generator SFT** — Supervised fine-tuning on high-Q episodes (Q >= 0.7) with Q-weighted duplication
2. **Generator DPO** — Direct Preference Optimization using Q-value preference pairs (high-Q chosen, low-Q rejected)
3. **Verifier DPO** — Separate DoRA adapter trained from base model to score code quality

**Requires:** Memory banks from eval notebook (`paperspace_bigcode.ipynb`). Episodes must be saved as
`/notebooks/CogMem/results/memory_bank_*.json`.

**Flow:**
- Cells 1-3: Setup (install deps, GPU check, load data)
- Cell 4: Generator SFT on high-Q episodes
- Cell 5: Generator DPO on Q-value preference pairs
- Cell 6: Verifier DPO (separate adapter from base model)
- Cell 7: Merge generator adapter into full model for Ollama
- Cell 8: **Restart kernel first!** Start Ollama + load models
- Cell 9: Evaluate on BigCodeBench-Hard (148 tasks)
- Cell 10: Package adapters for download

In [ ]:
# Cell 1: Install dependencies
!pip install "transformers==4.43.4" "peft==0.13.2" "accelerate==0.33.0" \
    "bitsandbytes==0.43.3" "datasets==2.20.0" "huggingface-hub>=0.24" \
    "pydantic>=2.0" "trl>=0.9.0" pyyaml scipy -q

# Verify critical imports
!python3 -c "import torch; print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"
!python3 -c "from transformers import Trainer; print('Trainer OK')"
!python3 -c "from trl import DPOTrainer; print('DPOTrainer OK')"

print("\n" + "="*60)
print("Restart kernel, then run Cell 2")
print("="*60)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepspeed 0.10.3 requires pydantic<2.0.0, but you have pydantic 2.12.5 which is incompatible.
torch 2.1.1+cu121, CUDA: True
2026-04-05 15:16:44.281632: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-05 15:16:44.318506: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-05 15:16:44.318613: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already be

In [ ]:
# Cell 2: GPU check + HuggingFace login
import torch
print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu}")
else:
    raise RuntimeError("No GPU detected — training requires CUDA")


import transformers, peft
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")

from huggingface_hub import login
import os
token = os.environ.get("HF_TOKEN", "")
if token:
    login(token=token)
    print("HF login OK (from env)")
else:
    print("Set HF_TOKEN env var or paste token below:")
    login()

torch: 2.1.1+cu121, CUDA: True
GPU: NVIDIA RTX A4000
transformers: 4.43.4
peft: 0.13.2
HF login OK (from env)


In [ ]:
# Cell 3: Clone CogMem, load memory banks, build training data
!cd /notebooks && git clone https://github.com/tungooxx/CogMem.git 2>/dev/null || \
    (cd /notebooks/CogMem && git pull && git checkout feat/bigcodebench-integration)
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

# ---- Model selection ----

MODEL_HF = "Qwen/Qwen2.5-3B-Instruct"
MODEL_OLLAMA = "qwen2.5:3b"

print(f"Model: {MODEL_HF}")
print(f"HF model:   {MODEL_HF}")
print(f"Ollama:     {MODEL_OLLAMA}")

# ---- Load memory banks ----
import json, glob
from pathlib import Path
from collections import defaultdict

mb_files = sorted(glob.glob("/notebooks/CogMem/results/memory_bank_*.json"))
if not mb_files:
    raise FileNotFoundError(
        "No memory banks found! Run eval notebook first.\n"
        "Expected: /notebooks/CogMem/results/memory_bank_*.json"
    )

all_episodes = []
for mb_path in mb_files:
    with open(mb_path, encoding="utf-8") as f:
        eps = json.load(f)
    passed = sum(1 for ep in eps if ep.get("success"))
    model = eps[0].get("model", Path(mb_path).stem) if eps else "?"
    print(f"  {Path(mb_path).name}: {len(eps)} episodes, {passed} passed ({passed/max(len(eps),1):.1%})")
    all_episodes.extend(eps)

print(f"\nTotal episodes loaded: {len(all_episodes)}")

# ---- Q-value triage stats ----
q_vals = [ep.get("q_value", 0.0) for ep in all_episodes]
high_q = [ep for ep in all_episodes if ep.get("q_value", 0.0) >= 0.7]
mid_q  = [ep for ep in all_episodes if 0.3 <= ep.get("q_value", 0.0) < 0.7]
low_q  = [ep for ep in all_episodes if ep.get("q_value", 0.0) < 0.3]

print(f"\nQ-value triage:")
print(f"  High  (Q >= 0.7):  {len(high_q):>4} episodes")
print(f"  Mid   (0.3-0.7):   {len(mid_q):>4} episodes")
print(f"  Low   (Q < 0.3):  {len(low_q):>4} episodes")

# ---- Build SFT data: high-Q episodes with Q-weighted duplication ----
from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT

sft_data = []
for ep in high_q:
    code = ep.get("generated_code") or ep.get("script", "")
    task_desc = ep.get("task_description", "")
    if not code or not task_desc:
        continue

    q = max(ep.get("q_value", 0.0), 0.01)
    copies = max(1, round(q * 3))  # Q=1.0 -> 3 copies, Q=0.7 -> 2

    for _ in range(copies):
        sft_data.append({"messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": task_desc},
            {"role": "assistant", "content": code},
        ]})

sft_path = "/notebooks/CogMem/results/sft_bigcode.jsonl"
Path(sft_path).parent.mkdir(parents=True, exist_ok=True)
with open(sft_path, "w", encoding="utf-8") as f:
    for item in sft_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\nSFT data: {len(sft_data)} examples (from {len(high_q)} high-Q episodes)")
print(f"  Saved to {sft_path}")

# ---- Build DPO pairs: same task_id, different Q-values ----
task_episodes = defaultdict(list)
for ep in all_episodes:
    tid = ep.get("task_id", "")
    if tid:
        task_episodes[tid].append(ep)

dpo_pairs = []
for tid, eps in task_episodes.items():
    if len(eps) < 2:
        continue
    # Sort by Q-value descending
    eps_sorted = sorted(eps, key=lambda e: e.get("q_value", 0.0), reverse=True)
    best = eps_sorted[0]
    worst = eps_sorted[-1]

    q_gap = best.get("q_value", 0.0) - worst.get("q_value", 0.0)
    if q_gap < 0.2:
        continue  # Not enough preference signal

    chosen_code = best.get("generated_code") or best.get("script", "")
    rejected_code = worst.get("generated_code") or worst.get("script", "")
    task_desc = best.get("task_description", "")

    if not chosen_code or not rejected_code or not task_desc:
        continue

    prompt = f"{SYSTEM_PROMPT}\n\n{task_desc}"
    dpo_pairs.append({
        "prompt": prompt,
        "chosen": chosen_code,
        "rejected": rejected_code,
    })

dpo_path = "/notebooks/CogMem/results/dpo_bigcode.jsonl"
with open(dpo_path, "w", encoding="utf-8") as f:
    for item in dpo_pairs:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"DPO pairs: {len(dpo_pairs)} (from {len(task_episodes)} unique tasks)")
print(f"  Saved to {dpo_path}")
print(f"\n{'='*60}")
print(f"Ready for training!")
print(f"  SFT examples: {len(sft_data)}")
print(f"  DPO pairs:    {len(dpo_pairs)}")

Already up to date.
Already on 'feat/bigcodebench-integration'
Your branch is up to date with 'origin/feat/bigcodebench-integration'.
Model: Qwen/Qwen2.5-3B-Instruct
HF model:   Qwen/Qwen2.5-3B-Instruct
Ollama:     qwen2.5:3b
  memory_bank_bigcode.json: 1140 episodes, 253 passed (22.2%)
  memory_bank_hard_qwen2.5_3b.json: 148 episodes, 10 passed (6.8%)
  memory_bank_qwen2_5_3b_full.json: 1140 episodes, 254 passed (22.3%)

Total episodes loaded: 2428

Q-value triage:
  High  (Q >= 0.7):   517 episodes
  Mid   (0.3-0.7):      0 episodes
  Low   (Q < 0.3):  1911 episodes

SFT data: 1551 examples (from 517 high-Q episodes)
  Saved to /notebooks/CogMem/results/sft_bigcode.jsonl
DPO pairs: 15 (from 1140 unique tasks)
  Saved to /notebooks/CogMem/results/dpo_bigcode.jsonl

Ready for training!
  SFT examples: 1551
  DPO pairs:    15


In [ ]:
!pip install "pydantic<2.0" -q
!pip install flash-attn --no-build-isolation


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 36.5 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 17.2 MB/s eta 0:00:00

^C
anceled
ERROR: Operation cancelled by user


In [ ]:
# Cell 4: Generator SFT training (DoRA on high-Q episodes)
CYCLE = 0  # Increment for each Q-STaR cycle

import json, gc, os
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    EarlyStoppingCallback,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq,
)


SFT_ADAPTER_DIR = f"/notebooks/CogMem/results/adapters/generator_sft_v{CYCLE}"
SFT_JSONL = "/notebooks/CogMem/results/sft_bigcode.jsonl"

# Load SFT data
with open(SFT_JSONL, encoding="utf-8") as f:
    sft_raw = [json.loads(line) for line in f if line.strip()]
print(f"SFT training samples: {len(sft_raw)}")

# 4-bit quantization
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_HF)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading {MODEL_HF} (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_HF,
    quantization_config=bnb_config,
    device_map={"":0},
    torch_dtype=torch.float16,
    attn_implementation="flash_attention_2",
)
model = prepare_model_for_kbit_training(model)

# DoRA config: r=16, alpha=32, targeting attention projections
dora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    use_dora=True,
)

gc.collect()
torch.cuda.empty_cache()

model = get_peft_model(model, dora_config)
model.print_trainable_parameters()

# Tokenize SFT data
def tokenize_sft(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    tok = tokenizer(text, truncation=True, max_length=2048, padding=False)
    tok["labels"] = tok["input_ids"].copy()
    return tok

dataset = Dataset.from_list(sft_raw).map(tokenize_sft, remove_columns=["messages"])
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
eval_ds = split["test"]
print(f"Tokenized: {len(dataset)} total -> {len(train_ds)} train, {len(eval_ds)} eval")

gc.collect()
torch.cuda.empty_cache()

# Training
os.makedirs(SFT_ADAPTER_DIR, exist_ok=True)


trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=SFT_ADAPTER_DIR,
        num_train_epochs=10,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,
        warmup_ratio=0.1,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=20,
        save_strategy="steps",

        save_steps=200,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        fp16=True,
        optim="paged_adamw_8bit",
        report_to="none",
        seed=42,
        gradient_checkpointing=True,
    ),
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Resume from checkpoint if exists
has_ckpt = (
    os.path.exists(SFT_ADAPTER_DIR)
    and any(d.startswith("checkpoint") for d in os.listdir(SFT_ADAPTER_DIR))
)
print(f"Starting SFT training (resume={has_ckpt})...")
trainer.train(resume_from_checkpoint=has_ckpt)

model.save_pretrained(SFT_ADAPTER_DIR)
tokenizer.save_pretrained(SFT_ADAPTER_DIR)

# Extract final loss
sft_final_loss = None
if trainer.state.log_history:
    for entry in reversed(trainer.state.log_history):
        if "loss" in entry:
            sft_final_loss = entry["loss"]
            break

with open(f"{SFT_ADAPTER_DIR}/training_log.json", "w") as f:
    json.dump({
        "type": "generator_sft",
        "cycle": CYCLE,
        "base_model": MODEL_HF,
        "dataset_size": len(dataset),
        "final_loss": sft_final_loss,
        "use_dora": True,
    }, f, indent=2)

print(f"\nSFT adapter saved to {SFT_ADAPTER_DIR}")
print(f"Final loss: {sft_final_loss}")

# Free VRAM
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print("VRAM freed.")

2026-04-05 15:14:16.194541: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-05 15:14:16.232646: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-05 15:14:16.232698: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-05 15:14:16.233850: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-05 15:14:16.240854: I tensorflow/core/platform/cpu_feature_guar

SFT training samples: 1551
Loading Qwen/Qwen2.5-3B-Instruct (4-bit)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 7,538,688 || all params: 3,093,477,376 || trainable%: 0.2437


Map:   0%|          | 0/1551 [00:00<?, ? examples/s]

Tokenized: 1551 total -> 1395 train, 156 eval
Starting SFT training (resume=False)...
[2026-04-05 15:14:28,226] [INFO] [real_accelerator.py:158:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_config.py:383: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'validate_by_name'
* 'validate_all' has been renamed to 'validate_default'
  warnings.warn(message, UserWarning)


AttributeError: 'FieldInfo' object has no attribute 'required'

In [ ]:
# Cell 5: Generator DPO training (builds on SFT adapter)
import json, gc, os
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import DPOConfig, DPOTrainer

os.environ["TRANSFORMERS_NO_FLASH_ATTENTION"] = "1"

DPO_JSONL = "/notebooks/CogMem/results/dpo_bigcode.jsonl"
SFT_ADAPTER_DIR = f"/notebooks/CogMem/results/adapters/generator_sft_v{CYCLE}"
GEN_DPO_DIR = f"/notebooks/CogMem/results/adapters/generator_v{CYCLE}"

# Load DPO pairs
with open(DPO_JSONL, encoding="utf-8") as f:
    dpo_raw = [json.loads(line) for line in f if line.strip()]

print(f"DPO pairs available: {len(dpo_raw)}")

if len(dpo_raw) < 10:
    print(f"\nSkipping DPO: only {len(dpo_raw)} pairs (need >= 10).")
    print("Generator will use SFT adapter only.")
    GEN_FINAL_DIR = SFT_ADAPTER_DIR
else:
    pref_dataset = Dataset.from_list(dpo_raw)
    print(f"DPO dataset: {len(pref_dataset)} pairs")

    # 4-bit quantization
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_HF)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load base model, then merge SFT adapter into it
    print(f"Loading {MODEL_HF} (4-bit)...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_HF,
        quantization_config=bnb_config,
        device_map={"":0},
        torch_dtype=torch.float16,
        attn_implementation="eager",
    )

    print(f"Merging SFT adapter from {SFT_ADAPTER_DIR}...")
    model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_DIR)
    model = model.merge_and_unload()
    model = prepare_model_for_kbit_training(model)

    gc.collect()
    torch.cuda.empty_cache()

    # New DoRA adapter for DPO on top of merged SFT
    dora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        use_dora=True,
    )

    os.makedirs(GEN_DPO_DIR, exist_ok=True)

    dpo_config = DPOConfig(
        output_dir=GEN_DPO_DIR,
        num_train_epochs=6,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=5e-5,
        beta=0.1,
        warmup_ratio=0.1,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=20,
        save_strategy="steps",
        save_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        bf16=True,
        seed=42,
        report_to="none",
        max_length=2048,
        max_prompt_length=512,
        gradient_checkpointing=True,
    )

    dpo_split = dpo_dataset.train_test_split(test_size=0.1, seed=42)
    dpo_train = dpo_split["train"]
    dpo_eval = dpo_split["test"]
    print(f"  DPO: {len(dpo_train)} train, {len(dpo_eval)} eval")

    trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        train_dataset=pref_dataset,
        processing_class=tokenizer,
        peft_config=dora_config,
    )

    print(f"Training generator DPO ({len(pref_dataset)} pairs)...")
    trainer.train()
    trainer.save_model(GEN_DPO_DIR)

    # Extract final loss
    dpo_final_loss = None
    if trainer.state.log_history:
        for entry in reversed(trainer.state.log_history):
            if "loss" in entry:
                dpo_final_loss = entry["loss"]
                break

    with open(f"{GEN_DPO_DIR}/training_log.json", "w") as f:
        json.dump({
            "type": "generator_sft_dpo",
            "cycle": CYCLE,
            "base_model": MODEL_HF,
            "sft_base": SFT_ADAPTER_DIR,
            "preference_pairs": len(pref_dataset),
            "dpo_beta": 0.1,
            "final_loss": dpo_final_loss,
            "use_dora": True,
        }, f, indent=2)

    print(f"\nGenerator DPO adapter saved to {GEN_DPO_DIR}")
    print(f"Final loss: {dpo_final_loss}")
    GEN_FINAL_DIR = GEN_DPO_DIR

    del model, trainer, base_model
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nGenerator final adapter: {GEN_FINAL_DIR}")

In [ ]:
# Cell 6: Verifier DPO training (from base model, NOT from SFT)
import json, gc, os
import torch
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import DPOConfig, DPOTrainer

os.environ["TRANSFORMERS_NO_FLASH_ATTENTION"] = "1"

DPO_JSONL = "/notebooks/CogMem/results/dpo_bigcode.jsonl"
VERIFIER_DIR = f"/notebooks/CogMem/results/adapters/verifier_v{CYCLE}"

# Load DPO pairs (same data, separate adapter)
with open(DPO_JSONL, encoding="utf-8") as f:
    dpo_raw = [json.loads(line) for line in f if line.strip()]

print(f"DPO pairs available: {len(dpo_raw)}")

if len(dpo_raw) < 10:
    print(f"\nSkipping verifier training: only {len(dpo_raw)} pairs (need >= 10).")
else:
    pref_dataset = Dataset.from_list(dpo_raw)
    print(f"Verifier DPO dataset: {len(pref_dataset)} pairs")

    # 4-bit quantization
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_HF)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load base model directly (verifier trains from scratch, not from SFT)
    print(f"Loading {MODEL_HF} (4-bit) for verifier...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_HF,
        quantization_config=bnb_config,
        device_map={"":0},
        torch_dtype=torch.float16,
        attn_implementation="eager",
    )
    model = prepare_model_for_kbit_training(model)

    gc.collect()
    torch.cuda.empty_cache()

    # Verifier DoRA config (same architecture as generator)
    dora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        use_dora=True,
    )

    os.makedirs(VERIFIER_DIR, exist_ok=True)

    dpo_config = DPOConfig(
        output_dir=VERIFIER_DIR,
        num_train_epochs=6,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=5e-5,
        beta=0.1,
        warmup_ratio=0.1,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=20,
        save_strategy="steps",
        save_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        bf16=True,
        seed=42,
        report_to="none",
        max_length=2048,
        max_prompt_length=512,
        gradient_checkpointing=True,
    )

    ver_split = dpo_dataset.train_test_split(test_size=0.1, seed=42)
    ver_train = ver_split["train"]
    ver_eval = ver_split["test"]
    print(f"  Verifier: {len(ver_train)} train, {len(ver_eval)} eval")

    trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        train_dataset=pref_dataset,
        processing_class=tokenizer,
        peft_config=dora_config,
    )

    print(f"Training verifier DPO ({len(pref_dataset)} pairs)...")
    trainer.train()
    trainer.save_model(VERIFIER_DIR)

    # Extract final loss
    ver_final_loss = None
    if trainer.state.log_history:
        for entry in reversed(trainer.state.log_history):
            if "loss" in entry:
                ver_final_loss = entry["loss"]
                break

    with open(f"{VERIFIER_DIR}/training_log.json", "w") as f:
        json.dump({
            "type": "verifier_dpo",
            "cycle": CYCLE,
            "base_model": MODEL_HF,
            "preference_pairs": len(pref_dataset),
            "dpo_beta": 0.1,
            "final_loss": ver_final_loss,
            "use_dora": True,
        }, f, indent=2)

    print(f"\nVerifier adapter saved to {VERIFIER_DIR}")
    print(f"Final loss: {ver_final_loss}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()
    print("VRAM freed.")

In [ ]:
# Cell 7: Merge generator adapter into full model for Ollama
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MERGED_DIR = "/notebooks/cogmem_merged"

# Use the best available generator adapter
# GEN_FINAL_DIR was set in Cell 5 (DPO if enough pairs, else SFT)
print(f"Merging adapter: {GEN_FINAL_DIR}")
print(f"Base model:      {MODEL_HF}")
print(f"Output:          {MERGED_DIR}")

# Load base model on CPU (full precision for merge)
print("\nLoading base model on CPU...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_HF, torch_dtype=torch.float16, device_map="cpu"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_HF)

# Load and merge adapter
print("Loading adapter...")
model = PeftModel.from_pretrained(model, GEN_FINAL_DIR)

print("Merging weights...")
model = model.merge_and_unload()

print("Saving merged model...")
model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")

# Create Ollama Modelfile with Qwen chat template
OLLAMA_MODEL_NAME = "cogmem-qwen-bigcode"
modelfile_path = "/notebooks/Modelfile.cogmem_bigcode"

with open(modelfile_path, "w") as f:
    f.write(f"""FROM {MERGED_DIR}
TEMPLATE \"\"\"{{{{if .System}}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{end}}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
\"\"\"
PARAMETER stop <|im_end|>
PARAMETER temperature 0
""")

# Install Ollama if needed and create model
!which ollama || (apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh)

import subprocess, time, os
try:
    subprocess.check_output(["ollama", "list"], timeout=5)
    print("\nOllama already running")
except Exception:
    proc = subprocess.Popen(
        ["ollama", "serve"],
        env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
        stdout=open("/tmp/ollama.log", "w"),
        stderr=subprocess.STDOUT,
    )
    time.sleep(5)

!ollama create {OLLAMA_MODEL_NAME} -f {modelfile_path}
!ollama list
print(f"\n{OLLAMA_MODEL_NAME} model created!")
print("\nNow restart kernel and run Cell 8 for evaluation.")

In [ ]:
# Cell 8: Restart kernel first! Start Ollama + load models
# IMPORTANT: After training, restart kernel to free GPU VRAM!

import subprocess, time, os

# Kill any existing Ollama process
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(2)

# Start Ollama server
proc = subprocess.Popen(
    ["ollama", "serve"],
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

!curl -sf http://localhost:11434/api/tags > /dev/null && echo "Ollama server OK" || echo "ERROR: Ollama not responding"

MODEL_OLLAMA = "qwen2.5:3b"

# Pull base model
print(f"\nPulling base model: {MODEL_OLLAMA}")
!ollama pull {MODEL_OLLAMA}

# Recreate CogMem model from merged weights
MERGED_DIR = "/notebooks/cogmem_merged"
OLLAMA_MODEL_NAME = "cogmem-qwen-bigcode"
modelfile_path = "/notebooks/Modelfile.cogmem_bigcode"

with open(modelfile_path, "w") as f:
    f.write(f"""FROM {MERGED_DIR}
TEMPLATE \"\"\"{{{{if .System}}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{end}}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
\"\"\"
PARAMETER stop <|im_end|>
PARAMETER temperature 0
""")

!ollama create {OLLAMA_MODEL_NAME} -f {modelfile_path}
!ollama list

# Smoke test
from openai import OpenAI
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

for model_name in [MODEL_OLLAMA, OLLAMA_MODEL_NAME]:
    resp = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": "Write a Python function that adds two numbers."}],
        max_tokens=200, temperature=0,
    )
    print(f"\n--- {model_name} ---")
    print(resp.choices[0].message.content[:300])

print("\nBoth models ready for evaluation!")

In [ ]:
# Cell 9: Evaluate on BigCodeBench-Hard (148 tasks)
# Compare base model vs CogMem model, resume-safe

import json, time, sys
from pathlib import Path
from openai import OpenAI
from datasets import load_dataset

if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

from cogmem.benchmarks.bigcodebench.prompts import format_messages, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Load BigCodeBench-Hard subset (148 tasks)
ds = load_dataset("bigcode/bigcodebench-hard", split="v0.1.4")
tasks = []
for item in ds:
    tasks.append({
        "task_id": item["task_id"],
        "instruct_prompt": item.get("instruct_prompt", ""),
        "complete_prompt": item.get("complete_prompt", ""),
        "test": item.get("test", ""),
        "canonical_solution": item.get("canonical_solution", ""),
        "entry_point": item.get("entry_point", ""),
    })

print(f"BigCodeBench-Hard: {len(tasks)} tasks")

MODEL_HF = "Qwen/Qwen2.5-3B-Instruct"
MODEL_OLLAMA = "qwen2.5:3b"

OLLAMA_MODEL_NAME = "cogmem-qwen-bigcode"
MODELS = [MODEL_OLLAMA, OLLAMA_MODEL_NAME]

results = {}

for model_name in MODELS:
    safe_name = model_name.replace(":", "_").replace("-", "_")
    checkpoint = f"/notebooks/eval_hard_{safe_name}.jsonl"

    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"Checkpoint: {checkpoint}")
    print(f"{'='*60}")

    # Resume from checkpoint
    done_ids = set()
    passed = 0
    total = 0

    if Path(checkpoint).exists():
        with open(checkpoint, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    ep = json.loads(line)
                    done_ids.add(ep["task_id"])
                    total += 1
                    if ep["passed"]:
                        passed += 1
        print(f"Resumed: {len(done_ids)} tasks already done")

    remaining = [t for t in tasks if t["task_id"] not in done_ids]
    start_time = time.time()

    for i, task in enumerate(remaining):
        try:
            messages = format_messages(task, use_instruct=True)
            resp = client.chat.completions.create(
                model=model_name, messages=messages,
                max_tokens=2048, temperature=0,
            )
            response = resp.choices[0].message.content
            code = extract_code(response)
            result = evaluate_solution(task, code, timeout=30, mode="subprocess")
            task_passed = result["passed"]
        except Exception as e:
            task_passed = False

        total += 1
        if task_passed:
            passed += 1

        # Save checkpoint
        with open(checkpoint, "a", encoding="utf-8") as f:
            f.write(json.dumps({"task_id": task["task_id"], "passed": task_passed}) + "\n")

        if (i + 1) % 10 == 0 or i < 3:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
            print(f"  [{total}/{len(tasks)}] Pass: {passed}/{total} ({passed/total:.1%}) | Rate: {rate:.0f}/hr")

    rate = passed / total if total > 0 else 0
    results[model_name] = {"passed": passed, "total": total, "rate": rate}
    print(f"\n  {model_name}: {passed}/{total} ({rate:.1%})")

# ---- Print comparison table ----
print(f"\n{'='*60}")
print(f"{'BigCodeBench-Hard RESULTS':^60}")
print(f"{'='*60}")
print(f"{'Model':<30} {'Passed':>8} {'Total':>8} {'Rate':>10}")
print(f"{'-'*60}")
for model_name, r in results.items():
    print(f"{model_name:<30} {r['passed']:>8} {r['total']:>8} {r['rate']:>9.1%}")

if len(results) == 2:
    rates = list(results.values())
    improvement = rates[1]["rate"] - rates[0]["rate"]
    print(f"\n{'Improvement':<30} {'':>8} {'':>8} {improvement:>+9.1%}")

# Save results
with open("/notebooks/bigcode_hard_eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved to /notebooks/bigcode_hard_eval_results.json")

In [ ]:
# Cell 10: Package adapters for download
import os

CYCLE = 0
ADAPTERS_DIR = "/notebooks/CogMem/results/adapters"

# Package all adapters from this cycle
adapter_dirs = []
for name in [f"generator_sft_v{CYCLE}", f"generator_v{CYCLE}", f"verifier_v{CYCLE}"]:
    path = os.path.join(ADAPTERS_DIR, name)
    if os.path.exists(path) and os.path.exists(os.path.join(path, "adapter_config.json")):
        adapter_dirs.append(name)
        print(f"  Found: {name}")

if not adapter_dirs:
    print("No adapters found to package.")
else:
    archive = f"/notebooks/cogmem_adapters_cycle{CYCLE}.tar.gz"
    dirs_str = " ".join(f"adapters/{d}" for d in adapter_dirs)
    !tar czf {archive} -C /notebooks/CogMem/results {dirs_str}
    !ls -lh {archive}
    print(f"\nDownload: {archive}")
    print(f"Contains: {', '.join(adapter_dirs)}")

# Also package eval results if available
for result_file in ["/notebooks/bigcode_hard_eval_results.json"]:
    if os.path.exists(result_file):
        !ls -lh {result_file}
        print(f"Download: {result_file}")